# GRPO: Group Relative Policy Optimization

> Implement GRPO for reasoning and mathematical problem solving

**GRPO Paper:** Shao et al., 2024 — *DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models*

**Key Differences from PPO:**
- **No value model** — Uses group mean baseline instead of critic
- **Group sampling** — Samples multiple responses per prompt
- **Relative rewards** — Rewards are relative to group average

**GRPO Objective:**
```
L_GRPO = E[ (1/G) Σ (min(π_θ/π_old * A, clip(π_θ/π_old) * A)) - β * KL(π_θ || π_ref) ]
```
Where advantage A = (r - mean(r_group)) / std(r_group)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Callable
from dataclasses import dataclass

## 1. Math Problem Dataset

In [ ]:
@dataclass
class MathProblem:
    """A mathematical reasoning problem"""
    question: str
    answer: float
    reasoning_steps: List[str]

math_problems = [
    MathProblem(
        question="If a train travels at 60 km/h for 2.5 hours, how far does it go?",
        answer=150.0,
        reasoning_steps=[
            "Distance = Speed × Time",
            "Distance = 60 km/h × 2.5 h",
            "Distance = 150 km"
        ]
    ),
    MathProblem(
        question="A rectangle has length 8 and width 5. What is its area?",
        answer=40.0,
        reasoning_steps=[
            "Area = Length × Width",
            "Area = 8 × 5",
            "Area = 40"
        ]
    ),
    MathProblem(
        question="What is 15% of 200?",
        answer=30.0,
        reasoning_steps=[
            "15% = 0.15",
            "0.15 × 200 = 30"
        ]
    ),
    MathProblem(
        question="If 3 apples cost $1.50, how much do 10 apples cost?",
        answer=5.0,
        reasoning_steps=[
            "Price per apple = $1.50 / 3 = $0.50",
            "10 apples = 10 × $0.50 = $5.00"
        ]
    ),
    MathProblem(
        question="Solve: 2x + 5 = 15",
        answer=5.0,
        reasoning_steps=[
            "2x = 15 - 5",
            "2x = 10",
            "x = 5"
        ]
    ),
]

print(f"Math problems: {len(math_problems)}")
for p in math_problems[:2]:
    print(f"\nQ: {p.question}")
    print(f"A: {p.answer}")
    print(f"Steps: {len(p.reasoning_steps)}")

## 2. Reward Function

In [ ]:
def extract_final_answer(text: str) -> float:
    """Extract numerical answer from generated text"""
    import re
    # Look for patterns like "= 150" or "Answer: 150"
    patterns = [
        r'=\s*([0-9]+\.?[0-9]*)',
        r'answer[s]?[:is]+\s*\$?([0-9]+\.?[0-9]*)',
        r'result[s]?[:is]+\s*\$?([0-9]+\.?[0-9]*)',
        r'([0-9]+\.?[0-9]*)\s*(km|miles|dollars|\$)?\s*$',
    ]
    
    for pattern in patterns:
        matches = re.findall(pattern, text.lower())
        if matches:
            try:
                return float(matches[-1] if isinstance(matches[-1], str) else matches[-1][0])
            except:
                continue
    
    # Fallback: find any number
    numbers = re.findall(r'[0-9]+\.?[0-9]*', text)
    if numbers:
        return float(numbers[-1])
    
    return 0.0

def compute_reward(generated: str, problem: MathProblem) -> float:
    """
    Compute reward for generated solution.
    Reward components:
    1. Answer correctness (0 or 1)
    2. Format bonus (has reasoning steps)
    3. Length penalty (not too short, not too long)
    """
    # Extract answer
    pred_answer = extract_final_answer(generated)
    
    # Correctness reward
    correct = abs(pred_answer - problem.answer) < 0.01
    correctness_reward = 1.0 if correct else 0.0
    
    # Format reward (has step-by-step reasoning)
    has_steps = ('step' in generated.lower() or 'first' in generated.lower() or
                 'then' in generated.lower() or generated.count('\n') >= 2)
    format_reward = 0.5 if has_steps else 0.0
    
    # Length penalty
    words = len(generated.split())
    if words < 10:
        length_penalty = -0.3  # Too short
    elif words > 200:
        length_penalty = -0.2  # Too long
    else:
        length_penalty = 0.0
    
    total_reward = correctness_reward + format_reward + length_penalty
    
    return max(total_reward, 0.0)

# Test reward function
test_response = """
Step 1: Distance = Speed × Time
Step 2: Distance = 60 km/h × 2.5 h
Step 3: Distance = 150 km
Answer: 150
"""

reward = compute_reward(test_response, math_problems[0])
print(f"Reward for good response: {reward:.2f}")

bad_response = "150"
reward_bad = compute_reward(bad_response, math_problems[0])
print(f"Reward for short response: {reward_bad:.2f}")

wrong_response = "The answer is 100 km"
reward_wrong = compute_reward(wrong_response, math_problems[0])
print(f"Reward for wrong response: {reward_wrong:.2f}")

## 3. Simple Policy Model

In [ ]:
class ReasoningPolicy:
    """
    Simplified policy for reasoning tasks.
    Generates responses based on learned patterns.
    """
    
    def __init__(self, vocab_size: int = 128, hidden_dim: int = 64):
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        
        # Simple network
        self.W1 = np.random.randn(vocab_size, hidden_dim) * 0.02
        self.W2 = np.random.randn(hidden_dim, vocab_size) * 0.02
        
        # Reasoning style bias
        self.reasoning_bias = np.zeros(vocab_size)
    
    def encode(self, text: str) -> np.ndarray:
        """Character encoding"""
        return np.array([ord(c) % self.vocab_size for c in text])
    
    def generate(self, prompt: str, max_length: int = 100, temperature: float = 1.0) -> str:
        """Generate a response"""
        tokens = self.encode(prompt)
        
        # Simple generation (deterministic for demo)
        h = np.mean(self.W1[tokens], axis=0)
        h = np.maximum(0, h)
        logits = h @ self.W2 + self.reasoning_bias
        
        # Generate based on learned patterns
        # For demo, use template-based generation
        responses = [
            "Step 1: Identify the given values\nStep 2: Apply the formula\nStep 3: Calculate the result\nAnswer: {ans}",
            "Let me solve this step by step.\nFirst, \nThen, \nTherefore, the answer is {ans}",
            "Using the formula: \nCalculation: \nFinal answer: {ans}",
            "{ans}",  # Short, no reasoning
        ]
        
        # Select based on policy state
        idx = int(np.abs(logits[0]) * 10) % len(responses)
        
        # Extract expected answer from prompt
        return responses[idx]
    
    def copy(self):
        new = ReasoningPolicy(self.vocab_size, self.hidden_dim)
        new.W1 = self.W1.copy()
        new.W2 = self.W2.copy()
        new.reasoning_bias = self.reasoning_bias.copy()
        return new
    
    def update(self, grads: Dict, lr: float = 0.01):
        self.W1 -= lr * grads.get('W1', 0)
        self.W2 -= lr * grads.get('W2', 0)
        self.reasoning_bias -= lr * grads.get('bias', 0)

# Initialize
policy = ReasoningPolicy()
reference = policy.copy()
old_policy = policy.copy()

print("Policy initialized")

## 4. GRPO Algorithm

In [ ]:
def grpo_update(
    policy: ReasoningPolicy,
    old_policy: ReasoningPolicy,
    reference: ReasoningPolicy,
    problem: MathProblem,
    group_size: int = 4,
    epsilon: float = 0.2,
    beta: float = 0.01,
    lr: float = 0.01
) -> Dict:
    """
    Perform one GRPO update step.
    
    1. Sample group of responses from old policy
    2. Compute rewards for each response
    3. Calculate advantages (relative to group mean)
    4. Update policy with clipped objective + KL penalty
    """
    
    # Step 1: Sample group of responses
    responses = []
    rewards = []
    
    for _ in range(group_size):
        resp = old_policy.generate(problem.question, max_length=100)
        # Fill in answer based on problem
        resp = resp.replace('{ans}', str(problem.answer))
        responses.append(resp)
        rewards.append(compute_reward(resp, problem))
    
    rewards = np.array(rewards)
    
    # Step 2: Compute group-relative advantages
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards) + 1e-8
    advantages = (rewards - mean_reward) / std_reward
    
    # Step 3: Compute policy loss
    # Simplified: use reward-weighted update
    weighted_advantage = np.mean(advantages * rewards)
    
    # Step 4: KL penalty (simplified)
    kl_penalty = beta * np.mean((policy.reasoning_bias - reference.reasoning_bias) ** 2)
    
    # Total objective
    objective = weighted_advantage - kl_penalty
    
    # Update policy (simplified gradient ascent)
    if objective > 0:
        # Increase reasoning bias for better responses
        policy.reasoning_bias += lr * objective * 0.1
    
    return {
        'responses': responses,
        'rewards': rewards,
        'mean_reward': mean_reward,
        'std_reward': std_reward,
        'advantages': advantages,
        'objective': objective,
        'kl_penalty': kl_penalty,
    }

# Test GRPO update
result = grpo_update(policy, old_policy, reference, math_problems[0], group_size=4)

print("GRPO Update Results:")
print(f"  Mean reward: {result['mean_reward']:.3f}")
print(f"  Std reward: {result['std_reward']:.3f}")
print(f"  Advantages: {result['advantages']}")
print(f"  Objective: {result['objective']:.4f}")
print(f"\nSample responses:")
for i, resp in enumerate(result['responses'][:2]):
    print(f"  {i+1}. Reward: {result['rewards'][i]:.2f} | {resp[:60]}...")

## 5. GRPO Training Loop

In [ ]:
# Training configuration
config = {
    'group_size': 4,
    'epsilon': 0.2,
    'beta': 0.01,
    'lr': 0.05,
    'epochs': 50,
}

# Metrics tracking
epoch_rewards = []
epoch_advantages = []
epoch_kl = []
correct_rate = []

print("Starting GRPO training...")
print("=" * 50)

for epoch in range(config['epochs']):
    epoch_reward_sum = 0
    epoch_adv_sum = 0
    epoch_kl_sum = 0
    correct_count = 0
    total_count = 0
    
    for problem in math_problems:
        # Update old policy
        old_policy = policy.copy()
        
        # GRPO update
        result = grpo_update(
            policy, old_policy, reference, problem,
            group_size=config['group_size'],
            epsilon=config['epsilon'],
            beta=config['beta'],
            lr=config['lr']
        )
        
        epoch_reward_sum += result['mean_reward']
        epoch_adv_sum += np.mean(np.abs(result['advantages']))
        epoch_kl_sum += result['kl_penalty']
        
        # Check correctness of best response
        best_idx = np.argmax(result['rewards'])
        best_resp = result['responses'][best_idx]
        pred = extract_final_answer(best_resp)
        if abs(pred - problem.answer) < 0.01:
            correct_count += 1
        total_count += 1
    
    # Record metrics
    epoch_rewards.append(epoch_reward_sum / len(math_problems))
    epoch_advantages.append(epoch_adv_sum / len(math_problems))
    epoch_kl.append(epoch_kl_sum / len(math_problems))
    correct_rate.append(correct_count / total_count)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:2d} | Avg Reward: {epoch_rewards[-1]:.3f} | "
              f"Accuracy: {correct_rate[-1]:.1%} | KL: {epoch_kl[-1]:.4f}")

print(f"\nFinal Accuracy: {correct_rate[-1]:.1%}")

## 6. Training Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Reward progression
axes[0, 0].plot(epoch_rewards, color='#FF6B6B', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Average Reward')
axes[0, 0].set_title('GRPO: Average Reward per Epoch')
axes[0, 0].grid(alpha=0.3)

# Accuracy
axes[0, 1].plot(correct_rate, color='#4ECDC4', linewidth=2, marker='o')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Problem Solving Accuracy')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].grid(alpha=0.3)

# Advantage magnitude
axes[1, 0].plot(epoch_advantages, color='#45B7D1', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Mean |Advantage|')
axes[1, 0].set_title('Advantage Magnitude (Group Variance)')
axes[1, 0].grid(alpha=0.3)

# KL divergence
axes[1, 1].plot(epoch_kl, color='#96CEB4', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('KL Penalty')
axes[1, 1].set_title('KL Divergence from Reference')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Group Sampling Analysis

In [ ]:
# Analyze effect of group size
group_sizes = [2, 4, 8, 16]
group_results = []

for gs in group_sizes:
    test_policy = ReasoningPolicy()
    test_ref = test_policy.copy()
    
    rewards_history = []
    for _ in range(20):
        for problem in math_problems:
            old = test_policy.copy()
            result = grpo_update(test_policy, old, test_ref, problem,
                               group_size=gs, lr=0.05)
            rewards_history.append(result['mean_reward'])
    
    group_results.append({
        'group_size': gs,
        'final_reward': np.mean(rewards_history[-10:]),
        'reward_std': np.std(rewards_history[-10:]),
    })

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))

sizes = [r['group_size'] for r in group_results]
rewards = [r['final_reward'] for r in group_results]
stds = [r['reward_std'] for r in group_results]

ax.bar(sizes, rewards, yerr=stds, capsize=5, color='#45B7D1',
       edgecolor='black', alpha=0.8)
ax.set_xlabel('Group Size (G)')
ax.set_ylabel('Final Average Reward')
ax.set_title('Effect of Group Size on GRPO Performance')
ax.set_xticks(sizes)
ax.grid(axis='y', alpha=0.3)

for s, r in zip(sizes, rewards):
    ax.text(s, r + 0.02, f'{r:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📊 Group Size Analysis:")
print(f"{'Group Size':<12} {'Final Reward':<15} {'Std Dev'}")
print("-" * 40)
for r in group_results:
    print(f"{r['group_size']:<12} {r['final_reward']:<15.3f} {r['reward_std']:.3f}")

## 8. GRPO vs PPO Comparison

In [ ]:
print("📊 GRPO vs PPO Comparison:")
print("=" * 60)

comparison = {
    'Feature': [
        'Value Model',
        'Baseline',
        'Memory',
        'Sample Efficiency',
        'Implementation',
        'Best For'
    ],
    'PPO': [
        'Required (critic)',
        'Value function output',
        'High (4 models)',
        'Moderate',
        'Complex',
        'General RL tasks'
    ],
    'GRPO': [
        'Not needed',
        'Group mean reward',
        'Low (2 models)',
        'High (group sampling)',
        'Simpler',
        'Reasoning, Math, Code'
    ]
}

for i in range(len(comparison['Feature'])):
    print(f"\n{comparison['Feature'][i]}:")
    print(f"  PPO:  {comparison['PPO'][i]}")
    print(f"  GRPO: {comparison['GRPO'][i]}")

# Memory comparison visualization
methods = ['PPO', 'GRPO']
memory = [4, 2]  # Number of models
colors = ['#FF6B6B', '#4ECDC4']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(methods, memory, color=colors, edgecolor='black', width=0.5)
ax.set_ylabel('Models in Memory')
ax.set_title('Memory Footprint: PPO vs GRPO')
ax.set_ylim(0, 5)

for bar, m in zip(bars, memory):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
           f'{m} models', ha='center', fontweight='bold')

# Add model labels
ax.text(0, 3.5, 'Policy\nValue\nReference\nReward', ha='center', fontsize=9, alpha=0.7)
ax.text(1, 2.5, 'Policy\nReference', ha='center', fontsize=9, alpha=0.7)

plt.tight_layout()
plt.show()

## 🎯 Exercises

1. **Reward Engineering**: Design better reward functions for math reasoning.
2. **Group Size**: Experiment with different group sizes and analyze variance.
3. **Length Penalty**: Implement dynamic length penalties based on problem complexity.
4. **Real Model**: Apply GRPO to a real model using `trl` or `verl` library.
5. **Chain-of-Thought**: Add explicit CoT reward for step-by-step reasoning.
6. **Rejection Sampling**: Combine GRPO with best-of-N rejection sampling.